# Notebook 1: Data Preparation for Science Fair Presentation Analyzer

This notebook downloads and processes public speech datasets for training our presentation analyzer.

**Datasets used:**
- **TED-LIUM 3** - TED talks (presentation-style speech)
- **LibriSpeech** - Audiobooks (clean speech baseline)
- **Common Voice 17** - Diverse accents (pronunciation robustness)
- **SPGISpeech** - Technical/academic vocabulary (domain adaptation)
- **VoxCeleb2** - Speaker diversity (prosody generalization)

**Output:** Processed datasets ready for fine-tuning in `data/processed/`

In [ ]:
# Install dependencies
!pip install -q datasets==2.18.0 huggingface_hub==0.22.2 librosa==0.10.2 torchaudio==2.3.0 torch==2.2.0 transformers==4.39.0 accelerate==0.29.0 jiwer==3.0.4

import os
import json
import tarfile
import zipfile
from pathlib import Path
from datasets import load_dataset, DatasetDict, Audio, load_from_disk, concatenate_datasets
import torchaudio
import librosa
import numpy as np
from tqdm.auto import tqdm

print("Dependencies installed")

In [ ]:
# Setup directories
BASE_DIR = Path("/content/speech_analyzer")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

for d in [RAW_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Base: {BASE_DIR}")
print(f"Raw: {RAW_DIR}")
print(f"Processed: {PROCESSED_DIR}")

## 1. TED-LIUM 3 (Primary - Presentation Style Speech)

In [ ]:
# Download TED-LIUM 3 via Hugging Face datasets
print("Loading TED-LIUM 3...")
tedlium = load_dataset("LIUM/tedlium", "release3", trust_remote_code=True)

# Cast audio column to 16kHz
tedlium = tedlium.cast_column("audio", Audio(sampling_rate=16000))

print(f"Train: {len(tedlium['train'])} samples")
print(f"Validation: {len(tedlium['validation'])} samples")
print(f"Test: {len(tedlium['test'])} samples")

# Save processed version
tedlium.save_to_disk(str(PROCESSED_DIR / "tedlium3"))
print("TED-LIUM 3 saved")

## 2. LibriSpeech (Clean Speech Baseline)

In [ ]:
# Load LibriSpeech (clean subsets)
print("Loading LibriSpeech...")
librispeech_train = load_dataset("librispeech_asr", "clean", split="train.100+train.360", trust_remote_code=True)
librispeech_val = load_dataset("librispeech_asr", "clean", split="validation", trust_remote_code=True)
librispeech_test = load_dataset("librispeech_asr", "clean", split="test", trust_remote_code=True)

librispeech = DatasetDict({
    "train": librispeech_train,
    "validation": librispeech_val,
    "test": librispeech_test
})

librispeech = librispeech.cast_column("audio", Audio(sampling_rate=16000))

print(f"Train: {len(librispeech['train'])} samples")
print(f"Validation: {len(librispeech['validation'])} samples")
print(f"Test: {len(librispeech['test'])} samples")

librispeech.save_to_disk(str(PROCESSED_DIR / "librispeech_clean"))
print("LibriSpeech saved")

## 3. Common Voice 17 (Accent Diversity)

In [ ]:
# Load Common Voice (English, validated only)
print("Loading Common Voice 17...")
cv = load_dataset("mozilla-foundation/common_voice_17_0", "en", split="train+validation+test", trust_remote_code=True)

# Filter validated only
cv = cv.filter(lambda x: x["accent"] is not None and len(x["accent"]) > 0)
cv = cv.filter(lambda x: x["locale"] == "en")

# Split
cv = cv.train_test_split(test_size=0.1, seed=42)
val_test = cv["test"].train_test_split(test_size=0.5, seed=42)

cv_dict = DatasetDict({
    "train": cv["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

cv_dict = cv_dict.cast_column("audio", Audio(sampling_rate=16000))

print(f"Train: {len(cv_dict['train'])} samples")
print(f"Validation: {len(cv_dict['validation'])} samples")
print(f"Test: {len(cv_dict['test'])} samples")

# Save subset (full is huge - take 50k for training)
cv_subset = cv_dict["train"].shuffle(seed=42).select(range(min(50000, len(cv_dict["train"]))))
cv_final = DatasetDict({
    "train": cv_subset,
    "validation": cv_dict["validation"].shuffle(seed=42).select(range(min(5000, len(cv_dict["validation"])))),
    "test": cv_dict["test"].shuffle(seed=42).select(range(min(5000, len(cv_dict["test"]))))
})

cv_final.save_to_disk(str(PROCESSED_DIR / "common_voice_en"))
print("Common Voice saved (subset)")

## 4. SPGISpeech (Technical/Academic Vocabulary)

In [ ]:
# SPGISpeech - parliamentary speeches with technical terms
print("Loading SPGISpeech...")
spgi = load_dataset("speechcolab/spgispeech", trust_remote_code=True)

# Split
spgi = spgi["train"].train_test_split(test_size=0.1, seed=42)
val_test = spgi["test"].train_test_split(test_size=0.5, seed=42)

spgi_dict = DatasetDict({
    "train": spgi["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

spgi_dict = spgi_dict.cast_column("audio", Audio(sampling_rate=16000))

print(f"Train: {len(spgi_dict['train'])} samples")
print(f"Validation: {len(spgi_dict['validation'])} samples")
print(f"Test: {len(spgi_dict['test'])} samples")

# Take subset for manageable training
spgi_subset = DatasetDict({
    "train": spgi_dict["train"].shuffle(seed=42).select(range(min(30000, len(spgi_dict["train"])))),
    "validation": spgi_dict["validation"].shuffle(seed=42).select(range(min(3000, len(spgi_dict["validation"])))),
    "test": spgi_dict["test"].shuffle(seed=42).select(range(min(3000, len(spgi_dict["test"]))))
})

spgi_subset.save_to_disk(str(PROCESSED_DIR / "spgispeech"))
print("SPGISpeech saved (subset)")

## 5. Create Combined Training Dataset

In [ ]:
# Combine datasets for Whisper fine-tuning
print("Creating combined dataset...")

# Load processed datasets
tedlium = load_from_disk(str(PROCESSED_DIR / "tedlium3"))
librispeech = load_from_disk(str(PROCESSED_DIR / "librispeech_clean"))
cv = load_from_disk(str(PROCESSED_DIR / "common_voice_en"))
spgi = load_from_disk(str(PROCESSED_DIR / "spgispeech"))

# Standardize columns (audio, text)
def standardize(ds):
    return ds.remove_columns([c for c in ds.column_names if c not in ["audio", "text", "sentence", "transcript"]])

for name, ds in [("tedlium", tedlium), ("librispeech", librispeech), ("cv", cv), ("spgi", spgi)]:
    # Handle different text column names
    text_col = "text" if "text" in ds["train"].column_names else ("sentence" if "sentence" in ds["train"].column_names else "transcript")
    if text_col != "text":
        ds = ds.rename_column(text_col, "text")
    ds = standardize(ds)
    globals()[name] = ds

# Combine train sets (weighted: more TED-LIUM + SPGI for presentation style)
train_combined = concatenate_datasets([
    tedlium["train"].shuffle(seed=42).select(range(min(50000, len(tedlium["train"])))),
    librispeech["train"].shuffle(seed=42).select(range(min(30000, len(librispeech["train"])))),
    cv["train"].shuffle(seed=42).select(range(min(20000, len(cv["train"])))),
    spgi["train"].shuffle(seed=42).select(range(min(30000, len(spgi["train"]))))
])

val_combined = concatenate_datasets([
    tedlium["validation"].shuffle(seed=42).select(range(min(5000, len(tedlium["validation"])))),
    librispeech["validation"].shuffle(seed=42).select(range(min(3000, len(librispeech["validation"])))),
    cv["validation"].shuffle(seed=42).select(range(min(2000, len(cv["validation"])))),
    spgi["validation"].shuffle(seed=42).select(range(min(3000, len(spgi["validation"]))))
])

test_combined = concatenate_datasets([
    tedlium["test"].shuffle(seed=42).select(range(min(5000, len(tedlium["test"])))),
    librispeech["test"].shuffle(seed=42).select(range(min(3000, len(librispeech["test"])))),
    cv["test"].shuffle(seed=42).select(range(min(2000, len(cv["test"])))),
    spgi["test"].shuffle(seed=42).select(range(min(3000, len(spgi["test"]))))
])

combined = DatasetDict({
    "train": train_combined.shuffle(seed=42),
    "validation": val_combined.shuffle(seed=42),
    "test": test_combined.shuffle(seed=42)
})

print(f"Combined Train: {len(combined['train'])}")
print(f"Combined Val: {len(combined['validation'])}")
print(f"Combined Test: {len(combined['test'])}")

combined.save_to_disk(str(PROCESSED_DIR / "combined_whisper"))
print("Combined dataset saved")

## 6. Extract Prosody Features (OpenSMILE)

In [ ]:
# Install OpenSMILE for prosody extraction
!pip install -q opensmile==3.0.2

import opensmile
import pandas as pd
from joblib import Parallel, delayed

# Initialize OpenSMILE with eGeMAPS feature set (standard for prosody)
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

print(f"Feature names ({len(smile.feature_names)}): {smile.feature_names[:10]}...")

In [ ]:
# Extract prosody features for a dataset split
def extract_prosody_features(dataset, split_name, max_samples=None):
    """Extract eGeMAPS features from audio samples."""
    features_list = []
    
    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))
    
    for i, sample in enumerate(tqdm(dataset, desc=f"Extracting {split_name}")):
        try:
            # Get audio array
            audio = sample["audio"]["array"]
            sr = sample["audio"]["sampling_rate"]
            
            # Extract features
            feats = smile.process_signal(audio, sr)
            
            # Convert to dict
            feat_dict = {}
            for idx, name in enumerate(smile.feature_names):
                feat_dict[name] = float(feats.iloc[0, idx])
            
            # Add metadata
            feat_dict["sample_id"] = i
            feat_dict["text"] = sample.get("text", sample.get("sentence", sample.get("transcript", "")))
            features_list.append(feat_dict)
            
        except Exception as e:
            print(f"Error on sample {i}: {e}")
            continue
    
    return pd.DataFrame(features_list)

# Extract for combined dataset (validation + test for now)
combined = load_from_disk(str(PROCESSED_DIR / "combined_whisper"))

print("Extracting prosody for validation...")
val_prosody = extract_prosody_features(combined["validation"], "validation", max_samples=5000)
val_prosody.to_parquet(PROCESSED_DIR / "prosody_validation.parquet")

print("Extracting prosody for test...")
test_prosody = extract_prosody_features(combined["test"], "test", max_samples=5000)
test_prosody.to_parquet(PROCESSED_DIR / "prosody_test.parquet")

print("Prosody features saved")
print(f"Validation shape: {val_prosody.shape}")
print(f"Test shape: {test_prosody.shape}")

## 7. Create Data Summary & Manifest

In [ ]:
# Create manifest for training
manifest = {
    "datasets": {
        "tedlium3": {
            "source": "LIUM/tedlium release3",
            "train_samples": len(tedlium["train"]),
            "val_samples": len(tedlium["validation"]),
            "test_samples": len(tedlium["test"]),
            "domain": "presentation_speech",
            "weight": 0.35
        },
        "librispeech_clean": {
            "source": "librispeech_asr clean",
            "train_samples": len(librispeech["train"]),
            "val_samples": len(librispeech["validation"]),
            "test_samples": len(librispeech["test"]),
            "domain": "clean_speech",
            "weight": 0.20
        },
        "common_voice_en": {
            "source": "mozilla-foundation/common_voice_17_0 en",
            "train_samples": len(cv["train"]),
            "val_samples": len(cv["validation"]),
            "test_samples": len(cv["test"]),
            "domain": "diverse_accents",
            "weight": 0.20
        },
        "spgispeech": {
            "source": "speechcolab/spgispeech",
            "train_samples": len(spgi["train"]),
            "val_samples": len(spgi["validation"]),
            "test_samples": len(spgi["test"]),
            "domain": "technical_academic",
            "weight": 0.25
        }
    },
    "combined": {
        "train_samples": len(combined["train"]),
        "val_samples": len(combined["validation"]),
        "test_samples": len(combined["test"]),
        "total_hours_estimate": len(combined["train"]) * 10 / 3600
    },
    "prosody_features": {
        "feature_set": "eGeMAPSv02",
        "n_features": len(smile.feature_names),
        "validation_samples": len(val_prosody),
        "test_samples": len(test_prosody)
    },
    "created_at": "2026-11-20",
    "version": "1.0"
}

with open(PROCESSED_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest saved")
print(json.dumps(manifest, indent=2))

## Data Preparation Complete!

**Next step:** Run `notebooks/02_whisper_finetune.ipynb` to fine-tune Whisper on the combined dataset.

**Files created in `/content/speech_analyzer/data/processed/`:**
- `tedlium3/` - TED-LIUM 3 dataset
- `librispeech_clean/` - LibriSpeech clean
- `common_voice_en/` - Common Voice English subset
- `spgispeech/` - SPGISpeech subset
- `combined_whisper/` - Combined dataset for Whisper fine-tuning
- `prosody_validation.parquet` - Prosody features (validation)
- `prosody_test.parquet` - Prosody features (test)
- `manifest.json` - Dataset manifest